In [0]:
import logging
from datetime import datetime
import pyspark.sql.functions as F

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


In [0]:
# data_quality_check
df_gold = spark.read.table("workspace.default.ppr_gold")

checks = {
    "gold_record_count": df_gold.count(),
    "null_prices": df_gold.filter(F.col("price").isNull()).count(),
    "null_counties": df_gold.filter(F.col("county").isNull()).count(),
    "min_year": df_gold.agg(F.min("year")).collect()[0][0],
    "max_year": df_gold.agg(F.max("year")).collect()[0][0]
}

for check, value in checks.items():
    logger.info(f"Quality check — {check}: {value}")

# Fail pipeline if data looks wrong
assert checks["gold_record_count"] > 700000, "Record count too low — pipeline may have failed"
assert checks["null_prices"] == 0, "Null prices found in Gold table"